In [3]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

def gradient_boosting_additive(X, y, n_estimators=10, learning_rate=0.1, max_depth=2):
    
    # Step 1: Initialize model
    F0 = np.mean(y)
    y_pred = np.full(len(y), F0)
    
    models = []
    gammas = []
    
    # Step 2: Additive iterations
    for m in range(n_estimators):
        
        # (a) Compute residuals (negative gradient)
        residuals = y - y_pred
        
        # (b) Fit weak learner
        model = DecisionTreeRegressor(max_depth=max_depth)
        model.fit(X, residuals)
        
        # (c) Predictions from weak learner
        h_m = model.predict(X)
        
        # (d) Compute optimal gamma (for MSE)
        gamma_m = np.sum(residuals * h_m) / np.sum(h_m ** 2)
        
        # (e) Update model (additive step)
        y_pred = y_pred + learning_rate * gamma_m * h_m
        
        # store
        models.append(model)
        gammas.append(gamma_m)
    
    return models, gammas, F0

In [4]:
def predict_gb(X, models, gammas, F0, learning_rate=0.1):
    y_pred = np.full(X.shape[0], F0)
    
    for model, gamma in zip(models, gammas):
        y_pred += learning_rate * gamma * model.predict(X)
    
    return y_pred

In [5]:
from sklearn.datasets import make_regression

# create data
X, y = make_regression(n_samples=100, n_features=1, noise=10, random_state=42)

# train
models, gammas, F0 = gradient_boosting_additive(X, y, n_estimators=5)

# predict
preds = predict_gb(X, models, gammas, F0)

print(preds[:5])

[ 18.01098395  -0.02166232 -15.63483634   2.64647324 -12.08748138]
